# Baseline, modelo final e extração da regra do Isolation Forest

Fecha a trilha de detecção de anomalias sobre as seis features de estilo do `01_Data_Prep`:

1. **Baselines**: a regra trivial "sem autor → alerta" e o Isolation Forest com os parâmetros padrão do scikit-learn.
2. **Modelo final**: Isolation Forest com 300 árvores (o mesmo do `02_IF_training`).
3. **Ablação**: o modelo final sem `tem_autor`, para medir quanto do resultado depende da autoria.
4. **Extração da regra**: árvore de decisão substituta ajustada nos alertas do detector (não no label).

Mesmo split e mesmo corte dos outros notebooks: treino só com `normalTrain` (True), limiar = percentil 95
dos scores de `normalValidation`, teste com 720 True e 1.800 Fake.

## Importando os dados preparados

In [1]:
import joblib

partitions, anomalyColumns = joblib.load("./dados_preparados.pkl")

## Imports

In [2]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.base import clone
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, export_text

from anomaly_utils import anomalyScores, binaryMetrics, combinePartitions, fitNormalOnly

RANDOM_STATE = 42
validationFrame = combinePartitions(partitions, "Validation")
testFrame = combinePartitions(partitions, "Test")
testLabels = testFrame["label"].to_numpy(dtype=int)

# Modelos

`evaluateDetector` segue o contrato do `anomaly_utils`: ajuste só com True, q95 de `normalValidation`
como limiar, métricas com Fake (1) como classe positiva. Acrescenta acurácia balanceada e macro-F1,
que não dependem da prevalência de 71% de Fake no teste.

In [3]:
def extendedMetrics(labels, scores, flags):
    metrics = binaryMetrics(labels, scores, flags)
    metrics["balancedAccuracy"] = balanced_accuracy_score(labels, flags)
    metrics["macroF1"] = f1_score(labels, flags, average="macro")
    return metrics


def evaluateDetector(pipeline, features):
    pipeline = fitNormalOnly(clone(pipeline), partitions["normalTrain"], features)
    threshold = np.quantile(anomalyScores(pipeline, partitions["normalValidation"], features), 0.95)
    scores = anomalyScores(pipeline, testFrame, features)
    return {
        "pipeline": pipeline, "features": features, "threshold": threshold,
        "metrics": extendedMetrics(testLabels, scores, scores >= threshold),
    }


def buildIsolationForest(**params):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("detector", IsolationForest(contamination="auto", random_state=RANDOM_STATE, n_jobs=-1, **params)),
    ])


featuresWithoutAuthor = [column for column in anomalyColumns if column != "tem_autor"]
detectors = {
    "IF padrão (100 árvores)": evaluateDetector(buildIsolationForest(), anomalyColumns),
    "IF final (300 árvores)": evaluateDetector(buildIsolationForest(n_estimators=300), anomalyColumns),
    "IF final sem tem_autor": evaluateDetector(buildIsolationForest(n_estimators=300), featuresWithoutAuthor),
}

## Baseline trivial: regra de autoria

No `01_Data_Prep`, `tem_autor` vale 1 em 97,97% das True e em 2,0% das Fake. A regra
"sem autor → alerta" não aprende nada e serve como piso: qualquer detector precisa superá-la
para mostrar que capta algo além da autoria.

In [4]:
authorRuleFlags = testFrame["tem_autor"].to_numpy() == 0
results = {"Regra: sem autor → alerta": extendedMetrics(testLabels, authorRuleFlags.astype(float), authorRuleFlags)}
results.update({name: detector["metrics"] for name, detector in detectors.items()})

metricColumns = ["rocAuc", "averagePrecision", "balancedAccuracy", "macroF1", "precision", "recall", "fpr", "tp", "fn", "fp", "tn"]
resultsTable = pd.DataFrame(results).T[metricColumns]
resultsTable.round(4)

,rocAuc,averagePrecision,balancedAccuracy,macroF1,precision,recall,fpr,tp,fn,fp,tn
Regra: sem autor → alerta,0.9835,0.9888,0.9835,0.9793,0.9938,0.9822,0.0153,1768.0,32.0,11.0,709.0
IF padrão (100 árvores),0.9819,0.9894,0.9782,0.9758,0.9894,0.9828,0.0264,1769.0,31.0,19.0,701.0
IF final (300 árvores),0.9807,0.9887,0.9768,0.9748,0.9883,0.9828,0.0292,1769.0,31.0,21.0,699.0
IF final sem tem_autor,0.5661,0.7668,0.5293,0.3324,0.8289,0.1211,0.0625,218.0,1582.0,45.0,675.0


# Extração da regra

A árvore substituta aprende a reproduzir o **alerta q95 do detector** a partir das mesmas features.
É ajustada na validação (imputação pela mediana do treino, a mesma do pipeline); a profundidade é
escolhida por validação cruzada de 5 folds dentro da validação (menor profundidade a até 0,01 da
melhor acurácia balanceada). O teste mede a fidelidade uma única vez.

In [5]:
MAX_DEPTHS = [1, 2, 3, 4]
DEPTH_TOLERANCE = 0.01
surrogateCv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

surrogates, fidelityRecords = {}, {}
for name in ["IF final (300 árvores)", "IF final sem tem_autor"]:
    detector = detectors[name]
    features, imputer = detector["features"], detector["pipeline"]["imputer"]
    validationInputs = pd.DataFrame(imputer.transform(validationFrame[features]), columns=features)
    testInputs = pd.DataFrame(imputer.transform(testFrame[features]), columns=features)
    validationAlerts = anomalyScores(detector["pipeline"], validationFrame, features) >= detector["threshold"]
    testAlerts = anomalyScores(detector["pipeline"], testFrame, features) >= detector["threshold"]

    depthScores = {
        depth: cross_val_score(
            DecisionTreeClassifier(max_depth=depth, min_samples_leaf=20, random_state=RANDOM_STATE),
            validationInputs, validationAlerts, cv=surrogateCv, scoring="balanced_accuracy",
        ).mean()
        for depth in MAX_DEPTHS
    }
    bestDepth = min(depth for depth, score in depthScores.items() if score >= max(depthScores.values()) - DEPTH_TOLERANCE)
    tree = DecisionTreeClassifier(max_depth=bestDepth, min_samples_leaf=20, random_state=RANDOM_STATE)
    tree.fit(validationInputs, validationAlerts)
    rulePredictions = tree.predict(testInputs)

    surrogates[name] = (tree, features)
    fidelityRecords[name] = {
        **{f"cvDepth{depth}": score for depth, score in depthScores.items()},
        "maxDepth": bestDepth, "leaves": tree.get_n_leaves(),
        "testFidelity": (rulePredictions == testAlerts).mean(),
        "testBalancedFidelity": balanced_accuracy_score(testAlerts, rulePredictions),
        "ruleBalancedAccuracyVsLabel": balanced_accuracy_score(testLabels, rulePredictions),
    }

pd.DataFrame(fidelityRecords).T.round(4)

,cvDepth1,cvDepth2,cvDepth3,cvDepth4,maxDepth,leaves,testFidelity,testBalancedFidelity,ruleBalancedAccuracyVsLabel
IF final (300 árvores),0.9964,0.9964,0.9964,0.9964,1.0,2.0,0.9956,0.9969,0.9835
IF final sem tem_autor,0.7098,0.8075,0.8581,0.9040,4.0,7.0,0.9718,0.9204,0.5240


## Regras extraídas

In [6]:
for name, (tree, features) in surrogates.items():
    display(Markdown(f"**{name}** (classe `True` = alerta)"))
    print(export_text(tree, feature_names=features, decimals=3))

**IF final (300 árvores)** (classe `True` = alerta)

|--- tem_autor <= 0.500
|   |--- class: True
|--- tem_autor >  0.500
|   |--- class: False



**IF final sem tem_autor** (classe `True` = alerta)

|--- typeTokenRatio <= 0.554
|   |--- typeTokenRatio <= 0.540
|   |   |--- class: True
|   |--- typeTokenRatio >  0.540
|   |   |--- class: True
|--- typeTokenRatio >  0.554
|   |--- uppercaseRatio <= 0.084
|   |   |--- punctuationDensity <= 0.279
|   |   |   |--- diversidade <= 0.938
|   |   |   |   |--- class: False
|   |   |   |--- diversidade >  0.938
|   |   |   |   |--- class: True
|   |   |--- punctuationDensity >  0.279
|   |   |   |--- class: True
|   |--- uppercaseRatio >  0.084
|   |   |--- uppercaseRatio <= 0.102
|   |   |   |--- class: True
|   |   |--- uppercaseRatio >  0.102
|   |   |   |--- class: True



## Conclusão

In [7]:
final, ablation, rule = (resultsTable.loc[name] for name in ["IF final (300 árvores)", "IF final sem tem_autor", "Regra: sem autor → alerta"])
finalFidelity = fidelityRecords["IF final (300 árvores)"]
display(Markdown(f"""
O Isolation Forest final obteve ROC-AUC **{final.rocAuc:.4f}**, acurácia balanceada **{final.balancedAccuracy:.4f}**,
recall Fake **{final.recall:.2%}** e FPR **{final.fpr:.2%}**. A regra trivial "sem autor → alerta" obteve acurácia
balanceada **{rule.balancedAccuracy:.4f}** e FPR **{rule.fpr:.2%}**.

A árvore substituta de profundidade {finalFidelity['maxDepth']} reproduz o detector com fidelidade
**{finalFidelity['testFidelity']:.2%}** no teste: a regra extraída é **"SE tem_autor = 0 → alerta"**.
Sem `tem_autor`, o mesmo detector cai para ROC-AUC **{ablation.rocAuc:.4f}** e recall **{ablation.recall:.2%}**.
O desempenho do detector vem da ausência de autor, um artefato de coleta do Fake.br, e não de um padrão de estilo.
"""))


O Isolation Forest final obteve ROC-AUC **0.9807**, acurácia balanceada **0.9768**,
recall Fake **98.28%** e FPR **2.92%**. A regra trivial "sem autor → alerta" obteve acurácia
balanceada **0.9835** e FPR **1.53%**.

A árvore substituta de profundidade 1 reproduz o detector com fidelidade
**99.56%** no teste: a regra extraída é **"SE tem_autor = 0 → alerta"**.
Sem `tem_autor`, o mesmo detector cai para ROC-AUC **0.5661** e recall **12.11%**.
O desempenho do detector vem da ausência de autor, um artefato de coleta do Fake.br, e não de um padrão de estilo.
